# 11 — Modelo Combinado con Stratified K-Fold Cross Validation (K=5)
Dataset: features de red + físicos concatenados (706.319 registros, 64 columnas)
Evaluación robusta: cada fold usa datos distintos como test, todos pasan por test una vez.

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    f1_score, precision_score, recall_score
)
import matplotlib.pyplot as plt

DELTA_COMBINED = "/Volumes/workspace/default/network_data/features_combined/"

df = spark.read.format("delta").load(DELTA_COMBINED)
print(f"Total registros : {df.count():,}")
print(f"Columnas        : {len(df.columns)}")

## 1 — Seleccionar features

In [0]:
# Excluimos columnas de metadatos — no son features del modelo
# Mismo criterio que en NB 08 y NB 12
exclude = [
    "window_id", "window_start", "window_end",
    "session_id", "label",
    # Features de ruido identificadas en NB 08
    "write_read_ratio", "min_payload_bytes",
    "max_payload_bytes", "write_read_ratio_safe",
    # Features temporales de laboratorio descartadas en NB 07
    "hour_01", "hour_02", "hour_03", "hour_04",
    "hour_05", "hour_06", "hour_07", "hour_08",
    "hour_09", "hour_10", "hour_11", "hour_22",
    "dow_mon", "dow_tue", "dow_wed", "dow_thu",
    "franja_manana", "franja_tarde", "franja_noche",
    # Ratios físicos descartados en NB 12
    "lit401_fit201_ratio", "fit101_fit201_ratio", "lit301_high",
]

feature_cols = [
    c for c in df.columns
    if c not in exclude
    and df.schema[c].dataType.simpleString() in
       ("double", "long", "int", "integer", "float")
]

print(f"Features seleccionadas : {len(feature_cols)}")
print(feature_cols)

## 2 — Preparar datos

In [0]:
# Traer a pandas — misma estrategia que en notebooks anteriores
pdf = (
    df.select(feature_cols + ["label"])
      .toPandas()
)
pdf[feature_cols] = pdf[feature_cols].fillna(0).astype(float)
pdf["label"]      = pdf["label"].astype(int)

X = pdf[feature_cols].values
y = pdf["label"].values

print(f"Shape      : {pdf.shape}")
print(f"Normal (0) : {(y==0).sum():,}  ({(y==0).mean()*100:.2f}%)")
print(f"Ataque (1) : {(y==1).sum():,}  ({(y==1).mean()*100:.2f}%)")

# Peso de clase — calculado sobre todo el dataset
# En cada fold se recalculará sobre el train de ese fold
n_normal = (y == 0).sum()
n_ataque = (y == 1).sum()
weight_global = round(n_normal / n_ataque, 2)
print(f"\nPeso clase Ataque (global) : {weight_global}")


## 3 — Stratified K-Fold Cross Validation (K=5)

In [0]:
# StratifiedKFold garantiza que cada fold mantiene
# la misma proporción 93/7 de clases que el dataset completo
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Acumuladores de métricas por fold
resultados = []

# Guardar predicciones de todos los folds para análisis global
all_y_test  = []
all_y_pred  = []
all_y_proba = []
all_idx     = []

print("Iniciando cross validation K=5...")
print("=============================")

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Peso de clase calculado sobre el train de este fold
    n_n = (y_train == 0).sum()
    n_a = (y_train == 1).sum()
    w_a = round(n_n / n_a, 2)

    # Entrenar modelo — mismos hiperparámetros
    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_leaf=5,
        class_weight={0: 1.0, 1: w_a},
        n_jobs=-1,
        random_state=42
    )
    rf.fit(X_train, y_train)

    # Predicciones
    y_pred  = rf.predict(X_test)
    y_proba = rf.predict_proba(X_test)[:, 1]

    # Métricas del fold
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    fold_res = {
        "fold":             fold,
        "auc_roc":          round(roc_auc_score(y_test, y_proba), 4),
        "auc_pr":           round(average_precision_score(y_test, y_proba), 4),
        "f1_ataque":        round(f1_score(y_test, y_pred, pos_label=1), 4),
        "precision_ataque": round(precision_score(y_test, y_pred, pos_label=1), 4),
        "recall_ataque":    round(recall_score(y_test, y_pred, pos_label=1), 4),
        "deteccion_%":      round(tp / (tp + fn) * 100, 2),
        "falsa_alarma_%":   round(fp / (fp + tn) * 100, 2),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
        "weight_ataque":    w_a
    }
    resultados.append(fold_res)

    # Acumular predicciones para análisis global posterior
    all_y_test.extend(y_test)
    all_y_pred.extend(y_pred)
    all_y_proba.extend(y_proba)
    all_idx.extend(test_idx)

    print(f"Fold {fold} | AUC-ROC={fold_res['auc_roc']} | "
          f"AUC-PR={fold_res['auc_pr']} | "
          f"Deteccion={fold_res['deteccion_%']}% | "
          f"Falsa alarma={fold_res['falsa_alarma_%']}%")

print("=" * 55)
print("Cross validation completado")

# Arrays globales
all_y_test  = np.array(all_y_test)
all_y_pred  = np.array(all_y_pred)
all_y_proba = np.array(all_y_proba)

## 4 — Resultados por fold y métricas globales

In [0]:
df_res = pd.DataFrame(resultados)

# Tabla por fold
print("Resultados por fold:")
print(df_res[[
    "fold", "auc_roc", "auc_pr",
    "f1_ataque", "deteccion_%", "falsa_alarma_%"
]].to_string(index=False))

# Media y desviación típica
print("\nMedia ± Desviación típica:")
metricas = ["auc_roc", "auc_pr", "f1_ataque", "deteccion_%", "falsa_alarma_%"]
for m in metricas:
    media = df_res[m].mean()
    std   = df_res[m].std()
    print(f"  {m:<20} : {media:.4f} ± {std:.4f}")

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Barras por fold — AUC-ROC y AUC-PR
x = np.arange(5)
w = 0.35
axes[0].bar(x - w/2, df_res["auc_roc"], w,
            color="#4C8BF5", alpha=0.85, label="AUC-ROC")
axes[0].bar(x + w/2, df_res["auc_pr"],  w,
            color="#E8453C", alpha=0.85, label="AUC-PR")
axes[0].axhline(df_res["auc_roc"].mean(), color="#4C8BF5",
                linestyle="--", lw=1, alpha=0.7,
                label=f"Media ROC ({df_res['auc_roc'].mean():.4f})")
axes[0].axhline(df_res["auc_pr"].mean(), color="#E8453C",
                linestyle="--", lw=1, alpha=0.7,
                label=f"Media PR ({df_res['auc_pr'].mean():.4f})")
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"Fold {i+1}" for i in range(5)])
axes[0].set_title("AUC-ROC y AUC-PR por fold", fontsize=12, fontweight="bold")
axes[0].set_ylim(0.9, 1.01)
axes[0].legend(fontsize=8)
axes[0].grid(axis="y", alpha=0.3)

# Detección y falsa alarma por fold
axes[1].bar(x - w/2, df_res["deteccion_%"], w,
            color="#059669", alpha=0.85, label="Deteccion (%)")
axes[1].bar(x + w/2, df_res["falsa_alarma_%"], w,
            color="#F59E0B", alpha=0.85, label="Falsa Alarma (%)")
axes[1].axhline(df_res["deteccion_%"].mean(), color="#059669",
                linestyle="--", lw=1, alpha=0.7,
                label=f"Media det. ({df_res['deteccion_%'].mean():.2f}%)")
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"Fold {i+1}" for i in range(5)])
axes[1].set_title("Deteccion y Falsa Alarma por fold",
                  fontsize=12, fontweight="bold")
axes[1].legend(fontsize=8)
axes[1].grid(axis="y", alpha=0.3)

plt.suptitle("Resultados K-Fold Cross Validation (K=5)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 5 — Análisis global (todos los folds combinados)

In [0]:
# Al combinar las predicciones de todos los folds
# cada registro del dataset ha pasado exactamente una vez por test
print("Metricas globales (todos los folds):")
print("=" * 50)
print(classification_report(
    all_y_test, all_y_pred,
    target_names=["Normal (0)", "Ataque (1)"]
))

cm_global = confusion_matrix(all_y_test, all_y_pred)
tn_g, fp_g, fn_g, tp_g = cm_global.ravel()

print(f"AUC-ROC global         : {roc_auc_score(all_y_test, all_y_proba):.4f}")
print(f"AUC-PR global          : {average_precision_score(all_y_test, all_y_proba):.4f}")
print("-" * 50)
print(f"Verdaderos Negativos   : {tn_g:,}")
print(f"Falsos Positivos       : {fp_g:,}")
print(f"Falsos Negativos       : {fn_g:,}")
print(f"Verdaderos Positivos   : {tp_g:,}")
print("-" * 50)
print(f"Tasa deteccion ataques : {tp_g/(tp_g+fn_g)*100:.2f}%")
print(f"Tasa falsa alarma      : {fp_g/(fp_g+tn_g)*100:.2f}%")
print("=" * 50)

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Matriz de confusión global
im = axes[0].imshow(cm_global, cmap="Blues")
plt.colorbar(im, ax=axes[0])
labels_cm = ["Normal (0)", "Ataque (1)"]
axes[0].set_xticks([0, 1]); axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(labels_cm)
axes[0].set_yticklabels(labels_cm)
axes[0].set_xlabel("Prediccion", fontsize=12)
axes[0].set_ylabel("Real", fontsize=12)
axes[0].set_title("Matriz de Confusion Global (K=5)",
                  fontsize=12, fontweight="bold")
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, f"{cm_global[i,j]:,}",
                     ha="center", va="center",
                     color="white" if cm_global[i,j] > cm_global.max()/2 else "black",
                     fontsize=14, fontweight="bold")

# Curva ROC global
from sklearn.metrics import roc_curve
fpr_g, tpr_g, _ = roc_curve(all_y_test, all_y_proba)
axes[1].plot(fpr_g, tpr_g, color="#4C8BF5", lw=2,
             label=f"AUC = {roc_auc_score(all_y_test, all_y_proba):.4f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("Tasa Falsos Positivos")
axes[1].set_ylabel("Tasa Verdaderos Positivos")
axes[1].set_title("Curva ROC Global (K=5)", fontsize=12, fontweight="bold")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6 — Importancia de features (último fold)

In [0]:
# La importancia del último fold es representativa
# Para una importancia más robusta se podría promediar entre folds
# pero el último fold es suficiente para identificar las features clave
importances = pd.DataFrame({
    "feature":    feature_cols,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("Top 20 features mas importantes:")
print(importances.head(20).to_string(index=False))

top20 = importances.head(20)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top20["feature"][::-1], top20["importance"][::-1],
        color="#4C8BF5", edgecolor="white")
ax.axvline(importances["importance"].mean(), color="gray",
           linestyle="--", lw=0.8,
           label=f"Media ({importances['importance'].mean():.4f})")
ax.set_title("Top 20 features — Importancia Random Forest (fold 5)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Importancia (Gini)")
ax.legend()
plt.tight_layout()
plt.show()

## 7 — Resumen final

In [0]:
print("=" * 60)
print("RESUMEN — MODELO COMBINADO K-FOLD (K=5)")
print("=" * 60)
print(f"  Dataset              : Red + Fisicos combinados")
print(f"  Total registros      : {len(X):,}")
print(f"  Features usadas      : {len(feature_cols)}")
print(f"  Folds                : 5 (Stratified, shuffle)")
print("-" * 60)
print("  Metricas por fold (media +- std):")
for m in metricas:
    media = df_res[m].mean()
    std   = df_res[m].std()
    print(f"    {m:<22} : {media:.4f} +- {std:.4f}")
print("-" * 60)
print("  Metricas globales (todos los folds):")
print(f"    AUC-ROC              : {roc_auc_score(all_y_test, all_y_proba):.4f}")
print(f"    AUC-PR               : {average_precision_score(all_y_test, all_y_proba):.4f}")
print(f"    Tasa deteccion       : {tp_g/(tp_g+fn_g)*100:.2f}%")
print(f"    Tasa falsa alarma    : {fp_g/(fp_g+tn_g)*100:.2f}%")
print(f"    Falsos Negativos     : {fn_g:,}")
print("-" * 60)
print(f"  Top feature            : {importances.iloc[0]['feature']}")
print(f"  Importancia top        : {importances.iloc[0]['importance']:.4f}")
print("=" * 60)